# Abstract section classification

Scientific abstracts are often structured with labeled sections, like "background" and "methods" ([example](https://pubmed.ncbi.nlm.nih.gov/1429477/)). In this homework, we will use Machine Learning methods to see if we can predict what section a sentence came from. We could use this to automatically label the sections.

Our dataset has four possible labels:

- 0: BACKGROUND
- 1: METHODS
- 2: RESULTS
- 3: CONCLUSIONS

First, we'll download the data and load it into a DataFrame.


In [1]:
!gdown 16BlMv_DRZz_3ybazTKY2H5OFU7kJtgCj

Downloading...
From: https://drive.google.com/uc?id=16BlMv_DRZz_3ybazTKY2H5OFU7kJtgCj
To: /content/data.tsv
100% 1.57M/1.57M [00:00<00:00, 31.5MB/s]


In [2]:
import pandas as pd

df = pd.read_csv('data.tsv', sep='\t')
df.head()

,label,text
0,BACKGROUND,"For unknown reasons, the incidence of sudden d..."
1,METHODS,We measured left ventricular monophasic action...
2,RESULTS,We demonstrated longer action potential durati...
3,RESULTS,"Also, BAY K 8644 produced phase 2 early afterd..."
4,RESULTS,"Phenylephrine, an alpha agonist, further incre..."


## 1. Feature-based Learning with Simple Machine Learning Methods

### Feature Engineering

Our goal is to train a model that can predict a label from the input texts. But models don't read words, they only take numbers as input. Therefore, people have developed a number of ways to convert raw inputs into numerical arrays. The process of transforming raw data into meaningful numerical representations (features) to help the model learn is called **feature engineering**.

One of the feature engineering methods in NLP text classification is Term Frequency - Inverse Document Frequency (TF-IDF). The method follows the intuition that
1. a word is more important in a document if it appears a lot in that document (TF)
2. a word is less informative if it appears in almost every document (IDF)

**1. (10pts) Use [`TfidfVectorizer`](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html) to convert the `text` column of `df` to bag-of-words features. Store the resulting NumPy `ndarray` in the variable `X`.**
- Hint: you can use the default parameters of `TfidfVectorizer`.

In [3]:
# YOUR CODE HERE (1)
from sklearn.feature_extraction.text import TfidfVectorizer
X = TfidfVectorizer().fit_transform(df['text']).toarray()

**2. (10pts) Use [`LabelEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) to convert the `label` column of `df` to integers. Store the resulting NumPy `ndarray` in the variable `y`.**

In [4]:
# YOUR CODE HERE (2)
from sklearn.preprocessing import LabelEncoder
y = LabelEncoder().fit_transform(df['label'])

### Data Splitting

To evaluate the model fairly and see whether it truly performs well on text classification, we will reserve 20% of the dataset as a test set, which will not be used during training. This test dataset, which the model has never seen in the training phase, serves as an unbiased tool for assessing the model's performance.

**3. (10pts) Use the Scikit-Learn `train_test_split()` function to split X and y into train and test sets, with 80% of the data as the training set, and store in `X_train`, `X_test`, `y_train`, `y_test`.**
  - Hint: check out the examples in the [documentation](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html).
  - Hint: look at the arguments `test_size` and `train_size`.

In [5]:
# YOUR CODE HERE (3)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

### Training

Now, we are ready to fit our data to a model for classification. At first, let's fit a Linear Support Vector Machine, or a Linear Support Vector Classifier (SVC). A linear SVC learns a linear decision boundary (a hyperplane) that separates classes while maximizing the margin, meaning it tries to keep the boundary as far as possible from the closest data points on each side. For data that are not perfectly separable, SVC allows some training points to be on the wrong side of the boundary using a soft-margin formulation.

**4. (10pts) Define `clf` as a [LinearSVC](https://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html) classifier and fit it to `X_train` and `y_train`.**

In [6]:
# YOUR CODE HERE (4)
from sklearn.svm import LinearSVC
clf = LinearSVC()
clf.fit(X_train, y_train)

LinearSVC()

### Evaluation

Now we have our trained SVC ready. Let's look at the performance.

In [7]:
from sklearn.metrics import accuracy_score
print('Training accuracy:', accuracy_score(y_train, clf.predict(X_train)))
print('Test accuracy:', accuracy_score(y_test, clf.predict(X_test)))

Training accuracy: 0.984375
Test accuracy: 0.7865


**5. (10pts) Try switch to another classifier model for classification. You can choose one from [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html), [KNeighborsClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html), [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html), [MLPClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html) or any other model of your choice.**

In [8]:
# YOUR CODE HERE (5)
from sklearn.ensemble import RandomForestClassifier
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

RandomForestClassifier()

In [9]:
print('Training accuracy:', accuracy_score(y_train, clf.predict(X_train)))
print('Test accuracy:', accuracy_score(y_test, clf.predict(X_test)))

Training accuracy: 1.0
Test accuracy: 0.7445


**Which model of your choice has the best test accuracy?**

YOUR ANSWER HERE (5):
It looks like LinearSVC has better test accuracy than RandomForestClassifier.

In the next section, we'll see if we can do better with deep learning.

## 2. Deep Learning with Pretrained transformers

Transformers can be pretrained on large amounts of unlabeled text by trying to predict the next word (as in GPT) or trying to predict missing words within a passage (as in BERT). This creates deep representations of each word (or token) that incorporate the surrounding context, making them extremely powerful for all kinds of downstream Natural Language Processing tasks. In this tutorial, we will use a pretrained BERT model, with a classification head.

We will use [HuggingFace](https://huggingface.co), a popular library and model repository that makes Deep Learning for NLP much easier. For more information about how to use HuggingFace transformers for classification, see https://huggingface.co/docs/transformers/tasks/sequence_classification.

First, we'll install some necessary packages and import libraries.


In [10]:
!pip install -q datasets evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00


In [12]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
# Removed create_optimizer from imports list under transformers since it should be handled automatically

Load the data, split into training and validation, and create HuggingFace `Dataset` objects to help with tokenization and batching:

In [13]:
from sklearn.model_selection import train_test_split

id2label = {0: "BACKGROUND", 1: "METHODS", 2: "RESULTS", 3: "CONCLUSIONS"}
label2id = {"BACKGROUND": 0, "METHODS": 1, "RESULTS": 2, "CONCLUSIONS": 3}
num_labels=4
df = pd.read_csv('data.tsv', sep='\t')
df['label'] = [label2id[x] for x in df['label']]
df_train, df_val = train_test_split(df, test_size=0.1, random_state=0)
train_ds = Dataset.from_pandas(df_train, split="train")
val_ds = Dataset.from_pandas(df_val, split="val")

Huggingface Datasets have columns like DataFrames. Inspect the first few rows:

In [14]:
train_ds[0:5]

{'label': [2, 2, 3, 3, 1],
 'text': ['Drug sequence had no effect on toxicities. ',
  'Four adult patients had transient hypotension. ',
  'Rimexolone has a low IOP-elevating potential, comparable to that of fluorometholone and less than that of dexamethasone sodium phosphate and prednisolone acetate.',
  'We believe that this is only the second reported case of acute cholestatic jaundice resulting from ciprofloxacin therapy. ',
  'In a Phase II trial, the authors evaluated the influence of paclitaxel, carboplatin, and an antimotility factor (acellular pertussis vaccine [APV]) in 18 patients with cisplatin- and methotrexate-resistant metastatic bladder carcinoma. '],
 '__index_level_0__': [1554, 2087, 5470, 2363, 7570]}

We'll load the pretrained [BiomedBERT](https://huggingface.co/microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract) model, a domain-adapted verison of the seminal [BERT](https://arxiv.org/pdf/1810.04805.pdf) model.

**6a. (10pts) Initialize an [`AutoTokenizer`](https://huggingface.co/docs/transformers/v4.35.2/en/model_doc/auto) named `tokenizer` using `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract` as the pretrained model path and using truncation to max model length.**
- Hint: truncation means dropping tokens beyond a certain position, to fit the model or otherwise. See [Padding and truncation](https://huggingface.co/docs/transformers/pad_truncation).

In [15]:
# YOUR CODE HERE (6a)
tokenizer = AutoTokenizer.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract", truncation=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

**6b. (10pts) Initialize an [AutoModelForSequenceClassification](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelforsequenceclassification) named `model`, again using `microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract` as the pretrained model path.**
- Hint: You will need to specify `num_labels` and the dictionaries `id2label` and `label2id`, which are defined above.

In [16]:
# YOUR CODE HERE (6b)
model = AutoModelForSequenceClassification.from_pretrained("microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract",
                                                           num_labels=num_labels,
                                                           id2label=id2label,
                                                           label2id=label2id)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:

It has warned us that the weights of the classification head were not loaded. This is expected, since the original model had a classification head that predicted missing tokens, whereas ours is predicting one of only four labels. This is okay--it's the rest of the model weights we care about! The warning suggests training the model, and that's exactly what we'll do.



### Tokenization

Text data needs to be "tokenized," which is the process of coverting raw strings into sequences of numbers that identify 'tokens' in our vocabulary. Tokens can be whole words, but some are punctuation or pieces of words. Having pieces of words in the vocabulary, let's handle words that were not in the pretraining data.

In [17]:
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

tokenized_train = train_ds.map(preprocess_function, batched=True)
tokenized_val = val_ds.map(preprocess_function, batched=True)

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

After tokenizing, our dataset has the additional fields `input_ids` (the list of tokens in each sentence) and `attention_mask` (which allows padding sentences to the same length):

In [18]:
tokenized_train[0]

{'label': 2,
 'text': 'Drug sequence had no effect on toxicities. ',
 '__index_level_0__': 1554,
 'input_ids': [2, 2445, 3377, 2105, 1982, 2224, 1755, 13981, 17, 3],
 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

`DataCollator` will handle batching for us.

In [19]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

In [20]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

**7. (20pts) Create a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.48.2/en/main_classes/trainer#transformers.TrainingArguments) object named `training_args` with:**
- 1 epoch
- a batch size of 16
- a learning rate of 1e-4
- storing the output in a directory named `section-model`
- `report_to=None` to avoid logging into Weights and Biases.

In [27]:
# YOUR CODE HERE (7)
training_args = TrainingArguments(
    num_train_epochs=1,
    per_device_train_batch_size=16,
    learning_rate=1e-4,
    output_dir="section-model",
    report_to=[]
)

**8. (20pts) Use a [`Trainer`](https://huggingface.co/docs/transformers/en/main_classes/trainer) named `trainer` to train `model` on `tokenized_train`.**
- pass in `training_args` from the last step
- use the `data_collator` as the data collator
- use `tokenized_train` as the training set
- use `tokenized_val` as the evaluation set
- use the `compute_metrics` function defined above to compute metrics during training
- call `.train()` and `.evaluate()` to train the model and evaluate the performance

In [28]:
# YOUR CODE HERE (8)
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics
)
trainer.train()
trainer.evaluate()

Step,Training Loss
500,0.368173


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': 0.38339751958847046,
 'eval_accuracy': 0.883,
 'eval_runtime': 3.7925,
 'eval_samples_per_second': 263.68,
 'eval_steps_per_second': 32.96,
 'epoch': 1.0}

This accuracy is not bad for a single epoch of a ten thousand training examples, and for a 4-class classification problem!

To use our model, we can use the convenient pipeline library, which wraps tokenization and model calls:

In [29]:
from transformers import pipeline

trainer.save_model("section-model")
classifier = pipeline("text-classification", model='section-model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Let's see what the model says about a few sentences (try your own!):

In [30]:
classifier("""Little work has been done in humans to evaluate
the potential benefit of potassium supplementation.""")

[{'label': 'BACKGROUND', 'score': 0.9571728110313416}]

In [31]:
classifier("""The mean MDS-UPDRS total score at baseline was 34.3 in
the deferiprone group and 33.2 in the placebo group and increased
(worsened) by 15.6 points and 6.3 points, respectively (difference,
9.3 points; 95% confidence interval, 6.3 to 12.2; P<0.001).""")

[{'label': 'RESULTS', 'score': 0.9830461144447327}]

In [32]:
classifier("""We conducted a multicenter, phase 2, randomized, double-blind
trial involving participants with newly diagnosed Parkinson's disease who
had never received levodopa.""")

[{'label': 'METHODS', 'score': 0.7783293724060059}]

In [33]:
classifier("""A device allowing the patient's abdominal viscera to hang
freely while the patient is in a prone position significantly reduces their
inferior vena caval pressure.""")

[{'label': 'CONCLUSIONS', 'score': 0.887324333190918}]

In [34]:
classifier("""The recent growth of this field, which began with initial studies
of yellow fever and seasonal influenza vaccines, has rapidly expanded to
include studies profiling responses to a range of vaccines and vaccine
platforms, including those targeting diverse pathogens and age groups""")

[{'label': 'BACKGROUND', 'score': 0.8938178420066833}]

In [35]:
classifier("""However, the yellow fever vaccine YF-17D induced a very distinct
response that had little or even negative correlation with responses to all
other vaccines, including other live viral vaccines such as varicella zoster
virus, HIV and Ebola""")

[{'label': 'RESULTS', 'score': 0.6839640140533447}]

In [37]:
classifier("""The concept of time-adjusted signatures based on vaccine-specific
response kinetics will be useful in the development of future signatures""")

[{'label': 'CONCLUSIONS', 'score': 0.9139139652252197}]